In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "/home/matis/code/Florian-Q/maraicherbio-prediction/notebooks/")

In [ ]:
import utils
import utils_series
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# --- Boucle : df_train / df_test pour chaque produit (split adaptatif, date fin commune) ---
df_all = utils.charger_dataframe()

# Définir la date de fin commune pour TOUS les splits
utils_series.GLOBAL_TEST_END_DATE = df_all['created'].max()
print(f'Date de fin commune : {utils_series.GLOBAL_TEST_END_DATE.date()}\n')

modeles = sorted(df_all['model'].unique())
print(f'{len(modeles)} produits à traiter\n')

dict_train = {}
dict_test = {}
skipped = []

for modele in modeles:
    try:
        df_produit = utils.charger_dataframe(modele)
        df_model = utils_series.complete_weekly_dataframe(df_produit, 'created', 'quantite_y')
        train, test = utils_series.split_adaptive_seasonal(df_model, test_pct=0.20)
        dict_train[modele] = train
        dict_test[modele] = test
    except ValueError as e:
        skipped.append(modele)
        continue

print(f'\nTerminé : {len(dict_train)} produits prêts  |  {len(skipped)} ignorés')
if skipped:
    print(f'Ignorés : {skipped}')

# Vérification
test_ends = [t.index.max().date() for t in dict_test.values()]
print(f'Toutes les fins de test = {test_ends[0]} ?  {len(set(test_ends)) == 1}')

Date de fin commune : 2026-05-27

100 produits à traiter

Train : 2014-06-08  →  2024-05-26  (521 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-07-06  →  2024-05-26  (517 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2014-06-29  →  2024-05-26  (518 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2015-01-11  →  2024-05-26  (490 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.4 ans (18%)
Train : 2014-05-11  →  2024-05-26  (525 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-01-05  →  2024-05-26  (543 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.4 ans (16%)
Train : 2014-02-23  →  2024-05-26  (536 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 a

In [ ]:
# ============================================================
# BASELINE v3 : Moyenne hebdo + split_adaptive + seasonal_metrics
# ============================================================
import numpy as np

results = []

for modele in dict_train.keys():
    y_train = dict_train[modele]['quantite_y']
    y_test = dict_test[modele]['quantite_y']

    # Baseline : moyenne par semaine ISO (1..53) sur toutes les années train
    weekly_avg = y_train.groupby(y_train.index.isocalendar().week).mean()
    y_pred = y_test.index.isocalendar().week.map(weekly_avg).fillna(0)
    y_pred = np.clip(y_pred.values, 0, None)

    metrics = utils_series.seasonal_metrics(y_test, y_pred)

    results.append({
        'produit': modele,
        'train_debut': y_train.index.min().strftime('%Y-%m'),
        'test_fin': y_test.index.max().strftime('%Y-%m'),
        'train_sem': len(y_train),
        'test_sem': len(y_test),
        'pct_zeros': round(metrics['pct_zeros'], 1),
        'MAE_in': round(metrics['MAE_in'], 2),
        'MAE_out': round(metrics['MAE_out'], 2),
        'MAE_all': round(metrics['MAE_all'], 2),
        'MAPE': round(metrics['MAPE'], 1),
        'sMAPE_all': round(metrics['sMAPE_all'], 1),
    })

# --- Tableau ---
df_base = pd.DataFrame(results).sort_values('sMAPE_all')
print(f'BASELINE v3 — {len(df_base)} produits (date fin commune : {df_base["test_fin"].iloc[0]})\n')
print(df_base.to_string(index=False))

# --- Résumé ---
print(f'\nMAE_all moyen : {df_base["MAE_all"].mean():.2f}  |  '
      f'MAE_in moyen : {df_base["MAE_in"].mean():.2f}  |  '
      f'MAE_out moyen : {df_base["MAE_out"].mean():.2f}')
print(f'sMAPE médian  : {df_base["sMAPE_all"].median():.1f} %  |  '
      f'Zéros moyen  : {df_base["pct_zeros"].mean():.1f} %')

# --- Top 5 / Bottom 5 ---
print(f'\n--- Top 5 (sMAPE) ---')
print(df_base[['produit', 'MAE_all', 'MAPE', 'sMAPE_all']].head(5).to_string(index=False))
print(f'\n--- Bottom 5 (sMAPE) ---')
print(df_base[['produit', 'MAE_all', 'MAPE', 'sMAPE_all']].tail(5).to_string(index=False))

BASELINE v3 — 93 produits (date fin commune : 2026-05)

                      produit train_debut test_fin  train_sem  test_sem  pct_zeros  MAE_in  MAE_out  MAE_all  MAPE  sMAPE_all
                        Prune     2019-07  2026-05        306        52       82.7    4.51     0.00     0.78  47.1       13.0
                   Petit pois     2014-06  2026-05        520       105       89.5    4.84     0.30     0.78  66.6       21.9
                 tomate verte     2014-10  2026-05        503       104       90.4    1.22     0.13     0.23  76.6       24.8
              Tomates cerises     2015-07  2026-05        462       104       67.3    1.99     0.08     0.70  61.5       26.6
              tomate ancienne     2019-08  2026-05        304        52       63.5    3.51     0.00     1.28  48.5       27.0
                    Courgette     2014-05  2026-05        525       105       50.5    5.13     0.03     2.55  37.1       27.4
         Jeune pousse epinard     2018-04  2026-05        320 

In [ ]:
df_base

,produit,train_debut,test_fin,train_sem,test_sem,pct_zeros,MAE_in,MAE_out,MAE_all,MAPE,sMAPE_all
67,Prune,2019-07,2026-05,254,104,86.5,3.74,0.06,0.56,68.8,14.3
92,tomate verte,2014-10,2026-05,450,157,91.7,1.24,0.12,0.21,76.9,21.2
54,Petit pois,2014-06,2026-05,467,158,89.2,4.36,0.31,0.75,80.4,21.7
78,Tomates cerises,2015-07,2026-05,409,157,66.9,2.16,0.05,0.75,73.3,25.6
35,Jeune pousse epinard,2018-04,2026-05,320,104,88.5,0.63,0.05,0.12,203.7,27.9
...,...,...,...,...,...,...,...,...,...,...,...
80,Vinaigre de cidre,2014-01,2026-05,488,158,53.2,0.63,0.89,0.77,43.9,131.3
32,Fontaine de Jus de pomme,2014-01,2026-05,487,158,57.0,0.80,1.29,1.08,65.7,133.7
39,Laurier,2014-02,2026-05,484,157,89.2,1.27,0.21,0.33,83.7,156.1
20,Cidre demi-sec,2014-02,2026-05,486,157,69.4,1.04,0.66,0.78,53.7,161.5


In [ ]:
# Ajouter la colonne weight_units (sans écraser df_base)
unit_map = df_all[['model', 'weight_units']].drop_duplicates().set_index('model')['weight_units']
df_base_units = df_base.copy()
df_base_units['weight_units'] = df_base_units['produit'].map(unit_map)
df_base_units

,produit,train_debut,test_fin,train_sem,test_sem,pct_zeros,MAE_in,MAE_out,MAE_all,MAPE,sMAPE_all,weight_units
67,Prune,2019-07,2026-05,306,52,82.7,4.51,0.00,0.78,47.1,13.0,kg
54,Petit pois,2014-06,2026-05,520,105,89.5,4.84,0.30,0.78,66.6,21.9,kg
92,tomate verte,2014-10,2026-05,503,104,90.4,1.22,0.13,0.23,76.6,24.8,kg
78,Tomates cerises,2015-07,2026-05,462,104,67.3,1.99,0.08,0.70,61.5,26.6,kg
91,tomate ancienne,2019-08,2026-05,304,52,63.5,3.51,0.00,1.28,48.5,27.0,kg
...,...,...,...,...,...,...,...,...,...,...,...,...
49,Pain campagne 500g,2018-02,2026-05,329,105,25.7,2.11,0.63,1.73,68.3,127.6,Pièce
32,Fontaine de Jus de pomme,2014-01,2026-05,540,105,57.1,0.78,1.22,1.03,66.0,135.7,Pièce
20,Cidre demi-sec,2014-02,2026-05,539,104,58.7,1.08,0.63,0.81,58.4,152.8,Pièce
39,Laurier,2014-02,2026-05,537,104,91.3,1.19,0.21,0.30,80.8,162.1,Botte
